# 🍎 실제 아두이노 센서 데이터 수집 및 Fine-tuning

이 노트북은 다음 작업을 수행합니다:
1. 아두이노 RGB 센서로 실제 과일 데이터 수집
2. 수집한 데이터를 CSV로 저장
3. 기존 모델을 센서 데이터로 Fine-tuning
4. 성능 비교 및 평가

**준비물:**
- 아두이노 + TCS34725 센서
- 실제 과일 3종 (Apple Red 1, Avocado, Blueberry)
- 기존 학습된 모델 (rgb_fruit_mlp.pth)

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset
import pandas as pd
import numpy as np
import serial
import time
from datetime import datetime
from pathlib import Path

In [2]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(777)
if device == 'cuda':
    torch.cuda.manual_seed_all(777)
print(f"Device: {device}")

Device: cuda


## 1단계: 기존 모델 클래스 정의

In [3]:
# 기존 MLP 모델 정의 (Fine-tuning용)
class RGBFruitMLP(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        return self.net(x)

print("✓ 모델 클래스 정의 완료")

✓ 모델 클래스 정의 완료


## 2단계: 실제 센서 데이터 수집

**사용 방법:**
1. 아두이노를 COM 포트에 연결
2. 실제 과일을 센서 앞에 위치
3. 아래 셀을 실행하여 각 과일별 데이터 수집
4. Enter 키로 측정값 저장, 'n'으로 건너뛰기, 'q'로 종료

In [5]:
# 시리얼 포트 설정
SERIAL_PORT = 'COM4'  # 아두이노 포트 확인 후 수정
BAUD_RATE = 9600

def collect_sensor_data(fruit_name, num_samples=50):
    """
    아두이노 센서로 실제 과일 RGB 값 수집
    
    Args:
        fruit_name: 과일 이름 (예: "Apple Red 1", "Avocado", "Blueberry")
        num_samples: 수집할 샘플 수
    
    Returns:
        DataFrame with columns: ['fruit', 'R', 'G', 'B', 'timestamp']
    """
    data = []
    
    print(f"\n{'='*60}")
    print(f"🍎 {fruit_name} 측정 시작")
    print(f"{'='*60}")
    print(f"목표: {num_samples}개 샘플 수집")
    print(f"현재: 0개 수집됨")
    print("\n📝 조작 방법:")
    print("  - Enter: 현재 측정값 저장")
    print("  - 'n' + Enter: 건너뛰기")
    print("  - 'q' + Enter: 종료")
    print(f"{'='*60}\n")
    
    try:
        ser = serial.Serial(SERIAL_PORT, BAUD_RATE, timeout=1)
        time.sleep(2)
        print(f"✓ {SERIAL_PORT} 연결 성공\n")
        
        last_rgb = None
        
        while len(data) < num_samples:
            if ser.in_waiting > 0:
                line = ser.readline().decode('utf-8').strip()
                
                try:
                    # RGB 값 파싱
                    if ',' in line and 'R:' not in line:
                        r, g, b = map(int, line.split(','))
                        last_rgb = (r, g, b)
                        
                        # 화면에 표시
                        print(f"\r📊 측정값: R={r:3d}, G={g:3d}, B={b:3d} | ", end='')
                        
                except ValueError:
                    continue
            
            # 사용자 입력 대기
            if last_rgb:
                user_input = input("저장? [Enter/n/q]: ").strip().lower()
                
                if user_input == 'q':
                    print("\n⚠️  수집 중단")
                    break
                elif user_input != 'n':
                    r, g, b = last_rgb
                    data.append({
                        'fruit': fruit_name,
                        'R': r,
                        'G': g,
                        'B': b,
                        'timestamp': datetime.now()
                    })
                    print(f"✓ 저장됨 ({len(data)}/{num_samples})\n")
                else:
                    print("건너뜀\n")
                
                last_rgb = None
            
            time.sleep(0.1)
        
        ser.close()
        print(f"\n{'='*60}")
        print(f"✅ {fruit_name} 수집 완료: {len(data)}개 샘플")
        print(f"{'='*60}\n")
        
    except serial.SerialException as e:
        print(f"\n❌ 시리얼 포트 에러: {e}")
        print(f"   {SERIAL_PORT}가 올바른지 확인하세요")
        return pd.DataFrame()
    
    except KeyboardInterrupt:
        print(f"\n⚠️  사용자에 의해 중단됨")
    except Exception as e:
        print(f"\n❌ 에러 발생: {e}")
    
    return pd.DataFrame(data)

print("✓ 데이터 수집 함수 정의 완료")

✓ 데이터 수집 함수 정의 완료


## 3단계: 과일별 데이터 수집 실행

**각 셀을 순서대로 실행하세요:**
- 한 과일 수집이 끝나면 다음 과일로 교체
- 다양한 각도와 조명에서 측정하면 더 좋습니다

In [11]:
# Apple Red 1 데이터 수집 (50개 샘플)
sensor_data_apple = collect_sensor_data("Apple Red 1", num_samples=50)

# 데이터 확인
if not sensor_data_apple.empty:
    print("\n📊 Apple Red 1 데이터 통계:")
    print(sensor_data_apple[['R', 'G', 'B']].describe())


🍎 Apple Red 1 측정 시작
목표: 50개 샘플 수집
현재: 0개 수집됨

📝 조작 방법:
  - Enter: 현재 측정값 저장
  - 'n' + Enter: 건너뛰기
  - 'q' + Enter: 종료

✓ COM4 연결 성공

📊 측정값: R=  2, G=  4, B=  3 | ✓ 저장됨 (1/50)

📊 측정값: R=  2, G=  4, B=  3 | ✓ 저장됨 (2/50)

📊 측정값: R=  2, G=  4, B=  3 | ✓ 저장됨 (3/50)

📊 측정값: R=  4, G=  2, B=  2 | ✓ 저장됨 (4/50)

📊 측정값: R=  3, G=  2, B=  2 | ✓ 저장됨 (5/50)

📊 측정값: R= 34, G= 14, B= 10 | ✓ 저장됨 (6/50)

📊 측정값: R= 31, G= 14, B=  9 | ✓ 저장됨 (7/50)

📊 측정값: R=  9, G=  4, B=  3 | ✓ 저장됨 (8/50)

📊 측정값: R= 36, G= 15, B= 10 | ✓ 저장됨 (9/50)

📊 측정값: R= 13, G=  5, B=  4 | ✓ 저장됨 (10/50)

📊 측정값: R= 46, G= 20, B= 14 | ✓ 저장됨 (11/50)

📊 측정값: R= 48, G= 19, B= 15 | ✓ 저장됨 (12/50)

📊 측정값: R= 42, G= 19, B= 13 | ✓ 저장됨 (13/50)

📊 측정값: R= 21, G=  9, B=  7 | ✓ 저장됨 (14/50)

📊 측정값: R= 24, G= 11, B=  8 | ✓ 저장됨 (15/50)

📊 측정값: R= 24, G= 11, B=  8 | ✓ 저장됨 (16/50)

📊 측정값: R= 36, G= 16, B= 12 | ✓ 저장됨 (17/50)

📊 측정값: R= 33, G= 15, B= 11 | ✓ 저장됨 (18/50)

📊 측정값: R= 39, G= 18, B= 13 | ✓ 저장됨 (19/50)

📊 측정값: R= 36, G= 17, B= 12 | ✓ 저장됨 (20/

In [15]:
# Avocado 데이터 수집 (50개 샘플)
sensor_data_avocado = collect_sensor_data("Avocado", num_samples=50)

# 데이터 확인
if not sensor_data_avocado.empty:
    print("\n📊 Avocado 데이터 통계:")
    print(sensor_data_avocado[['R', 'G', 'B']].describe())


🍎 Avocado 측정 시작
목표: 50개 샘플 수집
현재: 0개 수집됨

📝 조작 방법:
  - Enter: 현재 측정값 저장
  - 'n' + Enter: 건너뛰기
  - 'q' + Enter: 종료

✓ COM4 연결 성공


⚠️  사용자에 의해 중단됨


In [13]:
# Blueberry 데이터 수집 (50개 샘플)
sensor_data_blueberry = collect_sensor_data("Blueberry", num_samples=50)

# 데이터 확인
if not sensor_data_blueberry.empty:
    print("\n📊 Blueberry 데이터 통계:")
    print(sensor_data_blueberry[['R', 'G', 'B']].describe())


🍎 Blueberry 측정 시작
목표: 50개 샘플 수집
현재: 0개 수집됨

📝 조작 방법:
  - Enter: 현재 측정값 저장
  - 'n' + Enter: 건너뛰기
  - 'q' + Enter: 종료

✓ COM4 연결 성공

📊 측정값: R=  2, G=  4, B=  3 | ✓ 저장됨 (1/50)

📊 측정값: R=  2, G=  4, B=  3 | ✓ 저장됨 (2/50)

📊 측정값: R=  1, G=  2, B=  2 | ✓ 저장됨 (3/50)

📊 측정값: R=  2, G=  3, B=  3 | ✓ 저장됨 (4/50)

📊 측정값: R=  2, G=  3, B=  3 | 
⚠️  수집 중단

✅ Blueberry 수집 완료: 4개 샘플


📊 Blueberry 데이터 통계:
          R         G     B
count  4.00  4.000000  4.00
mean   1.75  3.250000  2.75
std    0.50  0.957427  0.50
min    1.00  2.000000  2.00
25%    1.75  2.750000  2.75
50%    2.00  3.500000  3.00
75%    2.00  4.000000  3.00
max    2.00  4.000000  3.00


## 4단계: 수집한 데이터 저장 및 확인

In [ ]:
# 모든 데이터 합치기
all_sensor_data = pd.concat([
    sensor_data_apple,
    sensor_data_avocado,
    sensor_data_blueberry
], ignore_index=True)

# CSV로 저장
csv_path = "real_sensor_data.csv"
all_sensor_data.to_csv(csv_path, index=False)

print(f"✅ 센서 데이터 저장 완료: {csv_path}")
print(f"\n📊 전체 데이터 요약:")
print(f"  총 샘플 수: {len(all_sensor_data)}")
print(f"\n클래스별 샘플 수:")
for fruit in all_sensor_data['fruit'].unique():
    count = len(all_sensor_data[all_sensor_data['fruit'] == fruit])
    print(f"  - {fruit}: {count}개")

print(f"\n📈 RGB 통계:")
print(all_sensor_data.groupby('fruit')[['R', 'G', 'B']].mean().round(1))

## 5단계: 센서 데이터용 Dataset 클래스

In [ ]:
class SensorFruitDataset(Dataset):
    """실제 센서에서 수집한 RGB 데이터용 Dataset"""
    
    def __init__(self, csv_path_or_df):
        """
        Args:
            csv_path_or_df: CSV 파일 경로 또는 pandas DataFrame
        """
        if isinstance(csv_path_or_df, str):
            self.df = pd.read_csv(csv_path_or_df)
        else:
            self.df = csv_path_or_df
        
        self.classes = sorted(self.df['fruit'].unique().tolist())
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        
        print(f"\n✓ 센서 데이터셋 로드:")
        print(f"  총 샘플: {len(self.df)}")
        print(f"  클래스 수: {len(self.classes)}")
        print(f"  클래스: {self.classes}")
        
        for cls in self.classes:
            count = len(self.df[self.df['fruit'] == cls])
            print(f"    - {cls}: {count}개")
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        
        # RGB 정규화 (0~255 → 0~1)
        r = row['R'] / 255.0
        g = row['G'] / 255.0
        b = row['B'] / 255.0
        
        features = torch.tensor([r, g, b], dtype=torch.float32)
        label = self.class_to_idx[row['fruit']]
        target = torch.tensor(label, dtype=torch.long)
        
        return features, target

print("✓ 센서 Dataset 클래스 정의 완료")

## 6단계: 기존 모델 로드 및 센서 데이터 준비

In [ ]:
# 기존 학습된 모델 로드
checkpoint = torch.load("rgb_fruit_mlp.pth", map_location=device)
model = RGBFruitMLP(num_classes=len(checkpoint['classes'])).to(device)
model.load_state_dict(checkpoint['model_state'])

print("✓ 기존 모델 로드 완료")
print(f"  클래스: {checkpoint['classes']}")

# 센서 데이터셋 생성
sensor_dataset = SensorFruitDataset("real_sensor_data.csv")

# Train/Validation 분할 (80:20)
train_size = int(0.8 * len(sensor_dataset))
val_size = len(sensor_dataset) - train_size

from torch.utils.data import random_split
sensor_train, sensor_val = random_split(
    sensor_dataset, 
    [train_size, val_size],
    generator=torch.Generator().manual_seed(777)
)

sensor_train_loader = DataLoader(sensor_train, batch_size=16, shuffle=True)
sensor_val_loader = DataLoader(sensor_val, batch_size=16, shuffle=False)

print(f"\n📊 데이터 분할:")
print(f"  학습 데이터: {train_size}개")
print(f"  검증 데이터: {val_size}개")

## 7단계: Fine-tuning 실행

**학습률을 낮춰서 기존 지식을 유지하면서 센서 데이터에 적응**

In [ ]:
# Fine-tuning 설정
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)  # 낮은 학습률

def evaluate_sensor(loader):
    """센서 데이터로 평가"""
    model.eval()
    total = 0
    correct = 0
    loss_sum = 0.0
    
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            out = model(x)
            loss = criterion(out, y)
            loss_sum += loss.item() * y.size(0)
            preds = out.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    
    return loss_sum / total, correct / total

# Fine-tuning 전 성능 확인
print("\n" + "="*60)
print("📊 Fine-tuning 전 센서 데이터 성능:")
print("="*60)
val_loss, val_acc = evaluate_sensor(sensor_val_loader)
print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_acc*100:.2f}%")
print("="*60)

In [ ]:
# Fine-tuning 실행
print("\n🔧 Fine-tuning 시작...\n")

num_epochs = 30
best_val_acc = 0.0

train_losses = []
val_losses = []
val_accs = []

for epoch in range(num_epochs):
    # 학습
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for x, y in sensor_train_loader:
        x = x.to(device)
        y = y.to(device)
        
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * y.size(0)
        preds = out.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)
    
    train_loss = running_loss / total
    train_acc = correct / total
    
    # 검증
    val_loss, val_acc = evaluate_sensor(sensor_val_loader)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    
    # Best 모델 저장
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "model_state": model.state_dict(),
            "classes": sensor_dataset.classes,
            "class_to_idx": sensor_dataset.class_to_idx,
            "val_acc": val_acc,
            "epoch": epoch + 1
        }, "rgb_fruit_mlp_finetuned_best.pth")
    
    # 로그 출력
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:02d}/{num_epochs} | "
              f"train_loss {train_loss:.4f} train_acc {train_acc*100:.2f}% | "
              f"val_loss {val_loss:.4f} val_acc {val_acc*100:.2f}%")

print(f"\n✅ Fine-tuning 완료!")
print(f"🏆 Best Validation Accuracy: {best_val_acc*100:.2f}%")

## 8단계: Fine-tuned 모델 저장 및 평가

In [ ]:
# 최종 모델 저장
final_state = {
    "model_state": model.state_dict(),
    "classes": sensor_dataset.classes,
    "class_to_idx": sensor_dataset.class_to_idx,
    "train_losses": train_losses,
    "val_losses": val_losses,
    "val_accs": val_accs,
    "final_val_acc": val_accs[-1]
}
torch.save(final_state, "rgb_fruit_mlp_finetuned.pth")

print("✅ Fine-tuned 모델 저장 완료!")
print(f"  파일: rgb_fruit_mlp_finetuned.pth")
print(f"  최종 검증 정확도: {val_accs[-1]*100:.2f}%")

# 클래스별 정확도
print("\n📊 클래스별 성능 평가:")
model.eval()
per_class_correct = [0 for _ in range(len(sensor_dataset.classes))]
per_class_total = [0 for _ in range(len(sensor_dataset.classes))]

with torch.no_grad():
    for x, y in sensor_val_loader:
        x = x.to(device)
        y = y.to(device)
        logits = model(x)
        preds = logits.argmax(1)
        
        for t, p in zip(y, preds):
            per_class_total[t.item()] += 1
            if t == p:
                per_class_correct[t.item()] += 1

print("\n결과:")
for cls, c_total, c_correct in zip(sensor_dataset.classes, per_class_total, per_class_correct):
    acc = 100.0 * c_correct / c_total if c_total > 0 else 0.0
    print(f"  {cls:20s}: {acc:5.2f}% ({c_correct}/{c_total})")

## 9단계: 실시간 테스트

**Fine-tuned 모델로 실시간 추론 테스트**

In [ ]:
# Fine-tuned 모델로 실시간 추론 테스트
print("🔄 Fine-tuned 모델로 실시간 추론 시작\n")

try:
    ser = serial.Serial(SERIAL_PORT, BAUD_RATE, timeout=1)
    time.sleep(2)
    print(f"✓ {SERIAL_PORT} 연결 성공")
    print("Press Ctrl+C to stop.\n")
    
    while True:
        if ser.in_waiting > 0:
            line = ser.readline().decode('utf-8').strip()
            
            try:
                if ',' in line and 'R:' not in line:
                    r, g, b = map(float, line.split(','))
                    
                    # 정규화
                    rgb_normalized = torch.tensor([r/255.0, g/255.0, b/255.0], 
                                                  dtype=torch.float32).to(device)
                    
                    # 추론
                    with torch.no_grad():
                        logits = model(rgb_normalized.unsqueeze(0))
                        probs = torch.softmax(logits, dim=1)[0]
                        pred_idx = logits.argmax(1).item()
                        pred_class = sensor_dataset.classes[pred_idx]
                        confidence = probs[pred_idx].item() * 100
                    
                    print(f"📊 RGB({int(r):3d}, {int(g):3d}, {int(b):3d}) → "
                          f"🍎 {pred_class} ({confidence:.1f}%)")
                    
                    # 아두이노로 결과 전송
                    ser.write(f"{pred_class}\n".encode('utf-8'))
                    
            except ValueError:
                continue
        
        time.sleep(0.1)

except serial.SerialException as e:
    print(f"\n❌ Serial error: {e}")
except KeyboardInterrupt:
    print("\n⚠️  사용자에 의해 중단됨")
finally:
    if 'ser' in locals() and ser.is_open:
        ser.close()
        print("✓ Serial port closed")

## 10단계: 학습 곡선 시각화 (선택)

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss 곡선
ax1.plot(train_losses, label='Train Loss', marker='o', markersize=3)
ax1.plot(val_losses, label='Val Loss', marker='s', markersize=3)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Fine-tuning Loss Curves')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy 곡선
ax2.plot([acc * 100 for acc in val_accs], label='Val Accuracy', 
         marker='o', markersize=3, color='green')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Fine-tuning Validation Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('finetuning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ 학습 곡선 저장: finetuning_curves.png")

## 📝 요약

**이 노트북으로 수행한 작업:**
1. ✅ 아두이노 센서로 실제 과일 RGB 데이터 수집
2. ✅ 수집한 데이터를 CSV로 저장
3. ✅ 기존 이미지 기반 모델을 센서 데이터로 Fine-tuning
4. ✅ Fine-tuned 모델 저장 및 평가
5. ✅ 실시간 추론 테스트

**생성된 파일:**
- `real_sensor_data.csv`: 수집한 센서 데이터
- `rgb_fruit_mlp_finetuned.pth`: Fine-tuned 모델
- `rgb_fruit_mlp_finetuned_best.pth`: Best 검증 정확도 모델
- `finetuning_curves.png`: 학습 곡선 그래프

**다음 단계:**
- 아두이노 프로젝트에서 Fine-tuned 모델 사용
- 더 많은 센서 데이터 수집하여 성능 개선
- 다양한 조명 조건에서 추가 데이터 수집